Building the District Livelihood Shares Table

In [4]:
import os

BRIGHT_FILE = "BRIGHT_clean_all_sheets.xlsx"
assert os.path.exists(BRIGHT_FILE), "Put BRIGHT_clean_all_sheets.xlsx in the Colab sidebar (Files panel) first."
print("Using:", BRIGHT_FILE)

Using: BRIGHT_clean_all_sheets.xlsx


## 1. Mapping table (assumptions)
This table is the only place where choices are made. Change a line here and re-run if the supervisor decides differently.
- Code **62** (shop sales, tailor, florist, postman…) → TRADE (mixed group, mostly shop/sales work)
- Fishing (codes 7–13, sector 2) → OTHER (no fishing category)
- Public servants, teachers, doctors, factory workers etc. → OTHER
- If the job code is 'Other' (999) or empty, the sector code is used instead.

In [5]:
import pandas as pd

CATS = ["AGRICULTURE & PLANTATION CROPS","LIVESTOCK & ANIMAL HUSBANDRY","FOOD PROCESSING & PRODUCTION","TRADE",
        "GARMENT TEXTILE, HANDICRAFTS","CONSTRUCTION, BUILDING AND REPAIR","TRANSPORT & LOGISTICS","BEAUTY AND WELLNESS","OTHER"]
A,L,F,T,G,C,TR,B,O = CATS

JOB = {}   # H1_03 job code -> category
for c in [1,2,3,4,5,6,111,112]:      JOB[c] = A    # farmers, farm managers, estate workers
for c in [14,15,16,18,19,20]:        JOB[c] = L    # livestock, vets, poultry, herders
for c in [21,22,23,24,25,26,27,58]:  JOB[c] = F    # food processing, butchers, bakers, cooks
for c in [28,29,30,31,32,33,62,106]: JOB[c] = T    # vendors, traders, shops (62 = assumption)
for c in [99,100,101,108]:           JOB[c] = G    # textile workers, artisans, weavers
for c in [42,43,44,45,46,47,48]:     JOB[c] = C    # construction trades
for c in [41,61,109,113]:            JOB[c] = TR   # drivers, informal transport, courier
for c in [59,60]:                    JOB[c] = B    # hairdresser, beautician, fitness

SECTOR = {1:A, 3:L, 4:F, 5:T, 7:C, 15:G}   # H1_02 sector, used only if job code is 999/empty

def to_cat(job, sector):
    if pd.notna(job) and job in JOB:
        return JOB[job]
    if (pd.isna(job) or job == 999) and pd.notna(sector) and sector in SECTOR:
        return SECTOR[sector]
    return O

# district code (a_02_district) -> name, same order as area_index.csv
DISTRICTS = ["Colombo","Gampaha","Kalutara","Galle","Hambantota","Matara","Jaffna","Kilinochchi","Mannar","Mullaitivu",
             "Vavuniya","Ampara","Batticaloa","Trincomalee","Anuradhapura","Polonnaruwa","Kandy","Matale","Nuwara Eliya",
             "Kurunegala","Puttalam","Badulla","Monaragala","Kegalle","Ratnapura"]
DIST = {i+1: n for i, n in enumerate(DISTRICTS)}

## 2. Read BRIGHT and count

In [6]:
hh  = pd.read_excel(BRIGHT_FILE, sheet_name="mod_a_household_identification", usecols=["hhcode","a_02_district"])
job = pd.read_excel(BRIGHT_FILE, sheet_name="mod_h1_primary_employment_long", usecols=["hhcode","hhmember","h1_01","h1_02","h1_03"])

# only people who work (status 1-7); not unemployed (8,9) or retired (10)
w = job[job.h1_01.isin(range(1, 8))].merge(hh, on="hhcode")
w["category"] = [to_cat(j, s) for j, s in zip(w.h1_03, w.h1_02)]
w["district"] = w.a_02_district.map(DIST)
print("Working people used:", len(w))
print(w.category.value_counts(normalize=True).round(3))

Working people used: 11535
category
OTHER                                0.554
AGRICULTURE & PLANTATION CROPS       0.141
TRADE                                0.083
CONSTRUCTION, BUILDING AND REPAIR    0.073
TRANSPORT & LOGISTICS                0.053
GARMENT TEXTILE, HANDICRAFTS         0.046
FOOD PROCESSING & PRODUCTION         0.036
LIVESTOCK & ANIMAL HUSBANDRY         0.008
BEAUTY AND WELLNESS                  0.006
Name: proportion, dtype: float64


In [7]:
counts = pd.crosstab(w.district, w.category).reindex(index=DISTRICTS, columns=CATS, fill_value=0)
shares = counts.div(counts.sum(axis=1), axis=0).round(4)
shares.insert(0, "n_workers", counts.sum(axis=1))
shares

category,n_workers,AGRICULTURE & PLANTATION CROPS,LIVESTOCK & ANIMAL HUSBANDRY,FOOD PROCESSING & PRODUCTION,TRADE,"GARMENT TEXTILE, HANDICRAFTS","CONSTRUCTION, BUILDING AND REPAIR",TRANSPORT & LOGISTICS,BEAUTY AND WELLNESS,OTHER
district,,,,,,,,,,
Colombo,1128,0.0567,0.0027,0.0461,0.1020,0.0745,0.0727,0.1135,0.0133,0.5186
Gampaha,1121,0.0330,0.0018,0.0473,0.1088,0.0419,0.0571,0.0660,0.0036,0.6405
Kalutara,633,0.0853,0.0047,0.0395,0.0885,0.0616,0.0932,0.0616,0.0016,0.5640
Galle,561,0.1907,0.0018,0.0392,0.0481,0.0588,0.0606,0.0339,0.0036,0.5633
Hambantota,315,0.1556,0.0095,0.0349,0.0698,0.0889,0.0825,0.0095,0.0032,0.5460
Matara,434,0.2235,0.0046,0.0230,0.1083,0.0530,0.0645,0.0461,0.0069,0.4700
Jaffna,682,0.0704,0.0029,0.0117,0.0381,0.0323,0.0909,0.0279,0.0015,0.7243
Kilinochchi,106,0.0566,0.0000,0.0377,0.0283,0.0566,0.0566,0.0000,0.0000,0.7642
Mannar,146,0.0616,0.0000,0.0205,0.0959,0.0890,0.0685,0.0205,0.0000,0.6438


## 3. Checks

In [8]:
assert len(shares) == 25
assert (shares[CATS].sum(axis=1).round(2) == 1.0).all()
print("Smallest district sample:", shares.n_workers.min(), "workers")
print(shares.loc[["Badulla","Monaragala"], CATS].T)

Smallest district sample: 106 workers
district                           Badulla  Monaragala
category                                              
AGRICULTURE & PLANTATION CROPS      0.2648      0.2476
LIVESTOCK & ANIMAL HUSBANDRY        0.0061      0.0195
FOOD PROCESSING & PRODUCTION        0.0305      0.0130
TRADE                               0.0733      0.0651
GARMENT TEXTILE, HANDICRAFTS        0.0204      0.0423
CONSTRUCTION, BUILDING AND REPAIR   0.0489      0.0456
TRANSPORT & LOGISTICS               0.0428      0.0163
BEAUTY AND WELLNESS                 0.0041      0.0065
OTHER                               0.5092      0.5440


## 4. Save

In [9]:
out = shares.reset_index().rename(columns={"index": "district"})
out.to_csv("district_livelihood_shares.csv", index=False)
print("Saved district_livelihood_shares.csv - right-click it in the sidebar to download.")

Saved district_livelihood_shares.csv - right-click it in the sidebar to download.
